# HM DeepSpatial V1：留出切片 1、2、3

选择服务器原来的 DeepSpatial Python 内核，按顺序运行全部单元格。

- 顺序运行三轮独立实验，每轮只留出一张切片，其余四张用于训练和完整体积重建。
- 每轮重新设置同一个随机种子并从头创建模型；留出切片在读取 H5AD 前被排除。
- 沿用 `configs/merfish_v1.json` 的全部训练、模型和生成参数。
- 每轮分别保存配置、checkpoint、最终模型和 `generated.h5ad`；已有输出目录不会被覆盖。
- 本 Notebook 只训练和生成，评价口径沿用现有分析流程。


In [1]:
import copy
import gc
import glob
import json
import os
from pathlib import Path
import re
import sys

# 支持从仓库根目录、HM_DeepSpital 或 examples 目录启动内核。
PROJECT = next(
    (candidate for parent in (Path.cwd(), *Path.cwd().parents)
     for candidate in (parent / "HM_DeepSpital", parent)
     if (candidate / "configs/merfish_v1.json").is_file()
     and (candidate / "deepspatial/core.py").is_file()),
    None,
)
if PROJECT is None:
    raise FileNotFoundError("请在 DeepSpatial 项目目录内打开此 Notebook。")
PROJECT = PROJECT.resolve()

HOLDOUT_SLICE_IDS = [1, 2, 3]
CONFIG_PATH = PROJECT / "configs/merfish_v1.json"
OUTPUT_ROOT = PROJECT / "outputs/v1_holdouts_1_2_3"

# 在导入 torch 前选择 GPU；如需修改 GPU，应重启内核后重新运行。
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
sys.path.insert(0, str(PROJECT))
if sys.platform == "win32" and (PROJECT / ".runtime_deps").is_dir():
    sys.path.insert(1, str(PROJECT / ".runtime_deps"))


In [2]:
import anndata as ad
import pytorch_lightning as pl
import torch
import deepspatial as ds

from examples.run_v1 import prepare_output

# 避免复用已从上游目录导入的同名模块。
if not Path(ds.__file__).resolve().is_relative_to(PROJECT):
    raise RuntimeError("当前内核已导入其他 deepspatial，请重启内核后运行全部单元格。")

print("实现：", ds.__file__)
print("CUDA_VISIBLE_DEVICES：", os.environ["CUDA_VISIBLE_DEVICES"])


实现： /private_sdb/huangyingjie/Code/DeepSpatial/HM_DeepSpital/deepspatial/__init__.py
CUDA_VISIBLE_DEVICES： 0


In [3]:
config = json.loads(CONFIG_PATH.read_text(encoding="utf-8-sig"))
paths = [Path(p) for p in glob.glob(str(PROJECT / config["input_glob"]), recursive=True)]
paths.sort(key=lambda p: [int(s) if s.isdigit() else s for s in re.split(r"(\d+)", p.name)])
depths = config["slice_z_um"]

if len(paths) != len(depths) or len(paths) < 2:
    raise ValueError("输入文件数必须与 slice_z_um 相同，且至少有两张切片。")
if (not HOLDOUT_SLICE_IDS
        or len(set(HOLDOUT_SLICE_IDS)) != len(HOLDOUT_SLICE_IDS)
        or any(type(i) is not int or i not in (1, 2, 3) or i >= len(paths)
               for i in HOLDOUT_SLICE_IDS)
        or len(paths) - 1 < 2):
    raise ValueError("留出编号应为 1、2、3 的非空、无重复子集，且每轮至少保留两张切片。")

OUTPUT_ROOT = OUTPUT_ROOT.resolve()
if not OUTPUT_ROOT.is_relative_to(PROJECT):
    raise ValueError("输出目录必须位于 HM_DeepSpital 内。")
existing = [OUTPUT_ROOT / f"holdout_{i}" for i in HOLDOUT_SLICE_IDS
            if (OUTPUT_ROOT / f"holdout_{i}").exists()]
if existing:
    raise FileExistsError(f"以下输出目录已存在，请在配置单元格更换 OUTPUT_ROOT：{existing}")

print("本次完整配置：")
print(json.dumps(config, ensure_ascii=False, indent=2))
for heldout_id in HOLDOUT_SLICE_IDS:
    train_ids = [i for i in range(len(paths)) if i != heldout_id]
    print(f"留出 {heldout_id}：训练 {train_ids}；输出 {OUTPUT_ROOT / f'holdout_{heldout_id}'}")


本次完整配置：
{
  "seed": 42,
  "input_glob": "../data/merfish_mouse_hypothalamus/**/merfish_[0-4].h5ad",
  "slice_z_um": [
    0,
    50,
    100,
    150,
    200
  ],
  "data": {
    "spatial_key": "spatial",
    "z_key": "z_coord",
    "label_key": "cell_class",
    "batch_size": 512,
    "num_workers": 0,
    "n_samples_base": 50000,
    "alpha_spatial": 0.5,
    "uot_reg": 0.05,
    "uot_tau": 0.05,
    "balanced_fraction": 0.2,
    "source_sampling": "stratified",
    "niche_config": {
      "radius": 50,
      "sigma": 25,
      "programs": {
        "myelin_panel_proxy": {
          "Mbp": 1,
          "Opalin": 1,
          "Ermn": 1,
          "Gjc3": 1
        }
      },
      "expression_transform": "log1p",
      "mode": "real"
    }
  },
  "model": {
    "patch_size": 8,
    "hidden_size": 256,
    "depth": 6,
    "num_heads": 8,
    "lambda_g": 0.1,
    "lambda_c": 10.0,
    "bidirectional_training": true
  },
  "training": {
    "max_epochs": 100,
    "accelerator": "auto",


## 顺序训练与生成

下面的单元格依次执行三轮完整训练。每轮沿用配置中的训练轮数、参数和完整体积生成方式，不把 `[1, 2, 3]` 同时从同一个模型的训练数据中排除。

若某轮失败，保留该轮已有输出和日志；更换输出目录后再重跑。


In [4]:
completed_runs = []

for heldout_id in HOLDOUT_SLICE_IDS:
    output = OUTPUT_ROOT / f"holdout_{heldout_id}"
    fold_config = copy.deepcopy(config)
    fold_config["run"] = {
        "implementation": str(Path(ds.__file__).resolve().relative_to(PROJECT)),
        "visible_files": [
            str(path.relative_to(PROJECT.parent))
            for index, path in enumerate(paths) if index != heldout_id
        ],
        "heldout_indices": [heldout_id],
        "z_source": "explicit_user_configuration",
    }
    prepare_output(output, fold_config)
    pl.seed_everything(fold_config.get("seed", 42), workers=True)
    print(f"开始留出 {heldout_id}：{output}", flush=True)

    observed = []
    model = None
    result = None
    try:
        for index, (path, depth) in enumerate(zip(paths, depths)):
            if index == heldout_id:
                continue
            adata = ad.read_h5ad(path)
            adata.obs[fold_config["data"].get("z_key", "z_coord")] = float(depth)
            observed.append(adata)
        # 切片保存在 observed 中，移除循环变量的额外引用。
        del adata

        model = ds.DeepSpatial()
        model.setup_data(observed, **fold_config["data"])
        model.build_model(**fold_config.get("model", {}))
        model.fit(
            save_dir=str(output / "checkpoints"),
            save_ckpt=True,
            **fold_config.get("training", {}),
        )
        if model.module.trainer.is_global_zero:
            model.save_model(output / "model")
            result = model.reconstruct_full_volume(
                observed, **fold_config.get("reconstruction", {}),
            )
            result.write_h5ad(output / "generated.h5ad")
            completed_runs.append({
                "heldout_id": heldout_id,
                "generated_cells": int(result.n_obs),
                "output": str(output),
            })
            print(f"留出 {heldout_id} 完成：生成 {result.n_obs} 个细胞", flush=True)
    finally:
        del result, model, observed
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


Seed set to 42


开始留出 1：/private_sdb/huangyingjie/Code/DeepSpatial/HM_DeepSpital/outputs/v1_holdouts_1_2_3/holdout_1


DeepSpatial: Building Trajectories: 100%|██████████| 3/3 [00:21<00:00,  7.26s/it]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
/private_sdb/huangyingjie/miniconda3/envs/deepspatial/lib/python3.10/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /private_sdb/huangyingjie/Code/DeepSpatial/HM_DeepSpital/outputs/v1_holdouts_1_2_3/holdout_1/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type | Params | Mode 
-------------------------------------------
0 | model     | GiT  | 7.7

Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.
DeepSpatial: 3D Reconstruct: 100%|██████████| 3/3 [03:51<00:00, 77.29s/gap, range=150.0-200.0um]


...

开始留出 3：/private_sdb/huangyingjie/Code/DeepSpatial/HM_DeepSpital/outputs/v1_holdouts_1_2_3/holdout_3


DeepSpatial: Building Trajectories: 100%|██████████| 3/3 [00:19<00:00,  6.57s/it]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/private_sdb/huangyingjie/miniconda3/envs/deepspatial/lib/python3.10/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /private_sdb/huangyingjie/Code/DeepSpatial/HM_DeepSpital/outputs/v1_holdouts_1_2_3/holdout_3/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type | Params | Mode 
-------------------------------------------
0 | model     | GiT  | 7.7 M  | train
1 | ema_model | GiT  | 7.7 M  | eval 
-------------------------------------------
7.7 M     Trainable params
7.7 M     Non-trainable params
15.4 M    Total params
61.735    Total estimated model params size (MB)
153       Modules in train mode
153       Modules in eval mode
/private_sdb/huangyingjie/miniconda3/envs/deepspatial/lib/python3.10/site-pac

Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.
DeepSpatial: 3D Reconstruct: 100%|██████████| 3/3 [03:44<00:00, 74.74s/gap, range=100.0-200.0um]


留出 3 完成：生成 113664 个细胞


In [5]:
for run in completed_runs:
    print(f"留出 {run['heldout_id']}：{run['generated_cells']} 个细胞；"
          f"{Path(run['output']) / 'generated.h5ad'}")


留出 1：114757 个细胞；/private_sdb/huangyingjie/Code/DeepSpatial/HM_DeepSpital/outputs/v1_holdouts_1_2_3/holdout_1/generated.h5ad
留出 2：112777 个细胞；/private_sdb/huangyingjie/Code/DeepSpatial/HM_DeepSpital/outputs/v1_holdouts_1_2_3/holdout_2/generated.h5ad
留出 3：113664 个细胞；/private_sdb/huangyingjie/Code/DeepSpatial/HM_DeepSpital/outputs/v1_holdouts_1_2_3/holdout_3/generated.h5ad
